<a href="https://colab.research.google.com/github/kaarthik-balakrishnan/SprindPOC_eyetracking/blob/main/notebooks/eye_tracking_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eye Tracking Pipeline

Complete pipeline for eye tracking from video to 3D gaze estimation.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Download repo
import os
import shutil
import zipfile

if os.path.exists('/content/SprindPOC_eyetracking'):
    shutil.rmtree('/content/SprindPOC_eyetracking')

print('Downloading repo...')
!wget -q https://github.com/kaarthik-balakrishnan/SprindPOC_eyetracking/archive/refs/heads/main.zip -O /tmp/repo.zip

print('Extracting...')
os.makedirs('/content/SprindPOC_eyetracking', exist_ok=True)
with zipfile.ZipFile('/tmp/repo.zip', 'r') as z:
    z.extractall('/content/')

temp_dir = '/content/SprindPOC_eyetracking-main'
for item in os.listdir(temp_dir):
    shutil.move(os.path.join(temp_dir, item), '/content/SprindPOC_eyetracking/')
shutil.rmtree(temp_dir)

%cd /content/SprindPOC_eyetracking
!pip install -r requirements.txt

## Configure Paths

In [ ]:
VIDEO_PATH = '/content/drive/MyDrive/EyeTracking/PXL_20260410_024928909.mp4'
OUTPUT_FOLDER = '/content/drive/MyDrive/EyeTracking/outputs'

import os
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

STABILIZED_PATH = os.path.join(OUTPUT_FOLDER, 'stabilized.mp4')
EYE_TRACKED_PATH = os.path.join(OUTPUT_FOLDER, 'eye_tracked.mp4')
PUPIL_DATA_PATH = os.path.join(OUTPUT_FOLDER, 'pupil_data.csv')
GAZE_3D_PATH = os.path.join(OUTPUT_FOLDER, 'gaze_3d.csv')

---

## Step 1: Video Stabilization

Stabilizes handheld camera movement using reference frame alignment.

In [ ]:
!python scripts/stabilize_video.py \
    --input {VIDEO_PATH} \
    --output {STABILIZED_PATH} \
    --ref-frames 5 \
    --smooth-radius 15 \
    --scale 0.25

---

## Step 2: Eye Tracking with Circle Overlay

Tracks eye position using reference frame alignment and draws a white circle around the eye center.

In [ ]:
!python scripts/eye_center.py \
    --input {STABILIZED_PATH} \
    --output {EYE_TRACKED_PATH} \
    --clip-limit 4.0 \
    --smooth-radius 5 \
    --num-points 12 \
    --circle-radius 25 \
    --scale 0.25

---

## Step 3: Pupil Tracking

Detects pupil position using HoughCircles.

In [ ]:
!python scripts/track_pupil.py \
    --input {EYE_TRACKED_PATH} \
    --output {PUPIL_DATA_PATH} \
    --min-radius 10 \
    --max-radius 80

---

## Step 4: 3D Sphere & Gaze Vector

Fits 3D sphere to pupil data and estimates gaze direction.

In [ ]:
!python scripts/gaze_3d.py \
    --input {PUPIL_DATA_PATH} \
    --output {GAZE_3D_PATH} \
    --smooth-window 5 \
    --sphere-radius 12.0 \
    --focal-length 500.0

---

## Results

Visualize the gaze data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

gaze_df = pd.read_csv(GAZE_3D_PATH)
print(gaze_df.head(20))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].plot(gaze_df['frame'], gaze_df['pupil_x'], label='X')
axes[0, 0].plot(gaze_df['frame'], gaze_df['pupil_y'], label='Y')
axes[0, 0].set_title('Pupil Position')
axes[0, 0].legend()
axes[0, 0].set_xlabel('Frame')

axes[0, 1].plot(gaze_df['frame'], gaze_df['pupil_radius'])
axes[0, 1].set_title('Pupil Radius')
axes[0, 1].set_xlabel('Frame')

axes[1, 0].plot(gaze_df['frame'], gaze_df['gaze_x'], label='X')
axes[1, 0].plot(gaze_df['frame'], gaze_df['gaze_y'], label='Y')
axes[1, 0].plot(gaze_df['frame'], gaze_df['gaze_z'], label='Z')
axes[1, 0].set_title('Gaze Direction Vector')
axes[1, 0].legend()
axes[1, 0].set_xlabel('Frame')

axes[1, 1].scatter(gaze_df['gaze_x'], gaze_df['gaze_y'], alpha=0.5, s=1)
axes[1, 1].set_title('Gaze Scatter (X vs Y)')
axes[1, 1].set_xlabel('Gaze X')
axes[1, 1].set_ylabel('Gaze Y')
axes[1, 1].set_aspect('equal')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, 'gaze_plots.png'), dpi=150)
plt.show()